![Logo del Proyecto](https://www.utdt.edu/Images/prensa/utdt-color-baja.png)

<h1 align="center"> Tesis - Master in Management + Analytics </h1>
<h2 align="center"> IA como Herramienta para el Control de Calidad de Imágenes<br> en el Diagnóstico de Retinopatía Diabética </h2>

**Fecha:** Octubre 2024<br>
**Autora:** Gabriela Moran<br>
**Tutor:** [Santiago Cisco](https://ar.linkedin.com/in/mariocisco)<br>

In [42]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
import configparser

In [43]:
MiM_path = Path(os.getcwd()).parent
DIR = sys.path.append(MiM_path/'data')
BASE = Path(MiM_path/'data')

# Cantidad de imágenes de mala calidad en cada dataset

In [49]:
with open(BASE/'labels/HRF.pkl','rb') as f:
    HRF = pickle.load(f)

with open(BASE/'labels/DDR.pkl','rb') as f:
    DDR = pickle.load(f)

with open(BASE/'labels/Kaggle_zhou.pkl','rb') as f:
    zhou = pickle.load(f)

with open(BASE/'labels/DRiD.pkl','rb') as f:
    drid = pickle.load(f)

In [50]:
HRF['dataset'] = 'HRF'
DDR['dataset'] = 'DDR'
zhou['dataset'] = 'Kaggle_Zhou'
drid['dataset'] = 'DRiD'

In [ ]:
todos = pd.concat([HRF,DDR,zhou,drid],ignore_index=True)

In [52]:
todos['label'] = todos.label.apply(lambda x: 'Mala' if x == 1 else 'Buena')

In [53]:
global_bin = configparser.ConfigParser()
global_bin.read(BASE/'splits/global_binaria.ini')

['c:\\Users\\gabim\\MiM\\data\\splits\\global_binaria.ini']

In [54]:
train = pd.DataFrame({'filename': global_bin['split'].get('training').split(sep=','), 'partition':'train'})
val = pd.DataFrame({'filename': global_bin['split'].get('validation').split(sep=','), 'partition':'val'})
test = pd.DataFrame({'filename': global_bin['split'].get('test').split(sep=','),'partition':'test'})

In [55]:
train['filename'] = train['filename'].str.split(pat='/',expand=True)[1]
val['filename'] = val['filename'].str.split(pat='/',expand=True)[1]
test['filename'] = test['filename'].str.split(pat='/',expand=True)[1]

In [56]:
partition = pd.concat([train,val,test],ignore_index=True)

In [57]:
todos['filename'] = todos['filename'].str.replace('\..*','', regex=True)

In [58]:
todos = todos.merge(partition, how='left',left_on = 'filename',right_on='filename')

In [59]:
resumen = todos.pivot_table(index='dataset',
                            columns=['partition','label'],
                            aggfunc={'partition':'count'},
                            fill_value= 0)

In [60]:
resumen.columns = resumen.columns.droplevel()

In [61]:
resumen.columns.set_names(['',''],inplace=True)

In [62]:
resumen

test       train          val     
             Buena Mala  Buena  Mala  Buena Mala
dataset                                         
DDR           3759  346   6260   575   2503  230
DRiD           758  842      0     0      0    0
HRF              3    3      9     9      6    6
Kaggle_Zhou  41797  873  33841  1285  10680  226

In [63]:
resumen['test','Total'] = resumen['test','Buena'] + resumen['test','Mala']
resumen['train','Total'] = resumen['train','Buena'] + resumen['train','Mala']
resumen['val','Total'] = resumen['val','Buena'] + resumen['val','Mala']

In [64]:
resumen = resumen.sort_index(axis=1)
resumen = resumen.reindex(level=0,columns=['train','val','test'])

In [65]:
resumen = pd.concat([resumen, pd.DataFrame(resumen.sum(axis=0)).T.rename(index={0: 'Total'})])

In [66]:
resumen['train','Buena'] = round(resumen['train','Buena']/resumen['train','Total'],2).fillna(0)
resumen['train','Mala'] = round(resumen['train','Mala']/resumen['train','Total'],2).fillna(0)

resumen['val','Buena'] = round(resumen['val','Buena']/resumen['val','Total'],2).fillna(0)
resumen['val','Mala'] = round(resumen['val','Mala']/resumen['val','Total'],2).fillna(0)

resumen['test','Buena'] = round(resumen['test','Buena']/resumen['test','Total'],2).fillna(0)
resumen['test','Mala'] = round(resumen['test','Mala']/resumen['test','Total'],2).fillna(0)


In [67]:
resumen

train                val               test             
            Buena  Mala  Total Buena  Mala  Total Buena  Mala  Total
DDR          0.92  0.08   6835  0.92  0.08   2733  0.92  0.08   4105
DRiD         0.00  0.00      0  0.00  0.00      0  0.47  0.53   1600
HRF          0.50  0.50     18  0.50  0.50     12  0.50  0.50      6
Kaggle_Zhou  0.96  0.04  35126  0.98  0.02  10906  0.98  0.02  42670
Total        0.96  0.04  41979  0.97  0.03  13651  0.96  0.04  48381

# Output

In [40]:
resumen.to_excel(BASE/'reports/resumen.xlsx')

In [41]:
resumen3.to_excel(BASE/'reports/resumen3.xlsx')